<a href="https://colab.research.google.com/github/rdianap/Classification-Data-Training/blob/main/Studi_Kasus_Klasifikasi_Breast_Cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Studi Kasus Klasifikasi: Diagnosis Kanker Payudara dengan Machine Learning

**Topik:** Penerapan Machine Learning untuk Klasifikasi dalam Kehidupan Nyata  
**Dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset (UCI Machine Learning Repository)

## **Permasalahan yang Dihadapi**
* **Latar Belakang:** Kanker payudara merupakan salah satu jenis kanker yang paling sering dialami perempuan. Benjolan pada payudara umumnya diperiksa dengan biopsi jarum halus (*Fine Needle Aspiration*/FNA), yaitu teknik yang murah dan minim invasif. Namun, hasil FNA dibaca secara visual oleh pemeriksa sehingga **bersifat subjektif**, dan sebagian kasus berakhir **tidak konklusif** (dalam literatur disebutkan hingga sekitar 25% hasil FNA; Teague dkk., 1997). Pasien dengan hasil seperti ini sering dirujuk ke biopsi bedah yang lebih mahal dan invasif.
* **Permasalahan Utama:** Adanya risiko kesalahan klasifikasi pada tahap awal diagnosis, yaitu *False Negative* (benjolan ganas terlewat dan dianggap jinak sehingga penanganan terlambat) dan *False Positive* (benjolan jinak dianggap ganas sehingga pasien menjalani tindakan yang sebenarnya tidak perlu).
* **Tujuan Machine Learning:** Membangun model klasifikasi biner (*binary classification*) berbasis 30 ciri inti sel yang dihitung dari citra digital sampel FNA untuk memprediksi secara objektif, cepat, dan akurat apakah suatu benjolan **ganas / *malignant* (1)** atau **jinak / *benign* (0)**.

## **Metode Klasifikasi yang Digunakan**
1. **Random Forest Classifier (Model Utama):**
   * Algoritma *ensemble learning* berbasis gabungan banyak pohon keputusan (*decision trees*).
   * **Keunggulan:** Mampu menangkap pola non-linear antar-fitur, relatif tahan terhadap *overfitting*, tidak sensitif terhadap skala fitur, serta dapat menghitung tingkat kepentingan tiap fitur (*Feature Importance*).
2. **Regresi Logistik / Logistic Regression (Model Pembanding):**
   * Algoritma klasifikasi parametrik berbasis fungsi sigmoid.
   * **Keunggulan:** Berfungsi sebagai *baseline* yang efisien, mudah diinterpretasi, dan menghasilkan estimasi probabilitas keganasan.

**Teknik pendukung:** `StandardScaler` (standardisasi fitur untuk Regresi Logistik), *Stratified train-test split* 80:20, dan `GridSearchCV` 5-fold untuk *hyperparameter tuning* Random Forest dengan metrik **recall** (agar kasus ganas yang terlewat ditekan seminimal mungkin).

> Catatan: pada penelitian aslinya, tim Universitas Wisconsin memakai *Multisurface Method-Tree* (pohon keputusan berbasis *linear programming*). Pada studi kasus ini dipakai algoritma yang lebih umum tersedia di `scikit-learn`.

## **Deskripsi Dataset**
* **Nama Dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset
* **Sumber Data:** UCI Machine Learning Repository (didonasikan 1995, lisensi CC BY 4.0): https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic
* **Link Direct Raw CSV (yang dipakai di notebook ini):** `https://github.com/rdianap/Classification-Data-Training/raw/refs/heads/main/Breast%20Cancer%20Wisconsin.csv`
* **Format File:** CSV dengan pemisah **titik koma (`;`)**, sehingga dibaca dengan `pd.read_csv(url, sep=";")`.
* **Jumlah Observasi:** 569 pasien (357 jinak dan 212 ganas), terdiri atas 30 fitur numerik, 1 kolom `id`, dan 1 variabel target. Tidak terdapat *missing value*.
* **Asal Fitur:** Dihitung dari citra digital sampel FNA massa payudara; menggambarkan karakteristik inti sel pada citra.

Setiap fitur dasar di bawah ini dihitung dalam **tiga versi**: `_mean` (rata-rata), `_se` (*standard error*), dan `_worst` (rata-rata dari tiga nilai terbesar), sehingga 10 fitur dasar × 3 = **30 fitur**.

### **Daftar Variabel:**
| Fitur Dasar | Tipe Data | Deskripsi | Contoh Nama Kolom |
| :--- | :--- | :--- | :--- |
| `radius` | Numerik | Rata-rata jarak dari pusat inti sel ke titik-titik pada tepinya | `radius_mean`, `radius_se`, `radius_worst` |
| `texture` | Numerik | Simpangan baku nilai skala abu-abu (*gray-scale*) | `texture_mean`, ... |
| `perimeter` | Numerik | Keliling inti sel | `perimeter_mean`, ... |
| `area` | Numerik | Luas inti sel | `area_mean`, ... |
| `smoothness` | Numerik | Variasi lokal panjang radius | `smoothness_mean`, ... |
| `compactness` | Numerik | Kekompakan bentuk: keliling² / luas − 1,0 | `compactness_mean`, ... |
| `concavity` | Numerik | Tingkat keparahan bagian cekung pada kontur | `concavity_mean`, ... |
| `concave points` | Numerik | Banyaknya bagian cekung pada kontur | `concave points_mean`, ... |
| `symmetry` | Numerik | Kesimetrisan inti sel | `symmetry_mean`, ... |
| `fractal_dimension` | Numerik | Dimensi fraktal ("pendekatan garis pantai" − 1) | `fractal_dimension_mean`, ... |
| `id` | Identitas | Nomor pasien (**dibuang** saat pemodelan) | `id` |
| **`diagnosis`** | **Binary Target** | **Hasil diagnosis akhir** | **M = Ganas (*Malignant*), B = Jinak (*Benign*)** |

**Inisialisasi Environment & Load Data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Set style visualisasi
sns.set_theme(style="whitegrid")

# Load Dataset dari GitHub (file memakai pemisah titik koma ';')
url = "https://github.com/rdianap/Classification-Data-Training/raw/refs/heads/main/Breast%20Cancer%20Wisconsin.csv"
df = pd.read_csv(url, sep=";")

# Jaga-jaga: buang kolom kosong sisa ekspor (jika ada)
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

print("=== 5 BARIS PERTAMA DATASET ===")
display(df.head())
print(f"\nDimensi Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")

**Eksplorasi Awal Data**

In [ ]:
# EKSPLORASI AWAL: DISTRIBUSI KELAS & STATISTIK DESKRIPTIF
print("=== DISTRIBUSI KELAS TARGET (diagnosis) ===")
print(df["diagnosis"].value_counts())
print(df["diagnosis"].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="diagnosis", data=df, hue="diagnosis", palette=["#2F855A", "#C53030"], legend=False, ax=axes[0])
axes[0].set_title("Distribusi Diagnosis (B = Jinak, M = Ganas)")

sns.boxplot(x="diagnosis", y="area_worst", data=df, hue="diagnosis", palette=["#2F855A", "#C53030"], legend=False, ax=axes[1])
axes[1].set_title("area_worst berdasarkan Diagnosis")
plt.tight_layout()
plt.show()

display(df.describe().T.head(10))

**Preprocessing Data**

In [ ]:
# PREPROCESSING DATA (MISSING VALUES, ENCODING, SPLIT, & SCALING)
print("=== PENGECEKAN MISSING VALUES ===")
print("Total missing value :", df.isnull().sum().sum())
print("Baris duplikat      :", df.duplicated().sum())

# Buang kolom ID (hanya identitas pasien, tidak punya makna prediktif)
df = df.drop(columns=["id"])

# Encoding target: M (Malignant/Ganas) = 1, B (Benign/Jinak) = 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

# Pemisahan Fitur (X) dan Target (y)
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

# Split Data Latih & Data Uji (80% Train, 20% Test) dengan Stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardisasi Fitur menggunakan StandardScaler
# (fit HANYA pada data latih agar tidak terjadi kebocoran data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nUkuran Data Latih : {X_train.shape[0]} baris")
print(f"Ukuran Data Uji   : {X_test.shape[0]} baris")
print(f"Proporsi Ganas (latih/uji): {y_train.mean():.3f} / {y_test.mean():.3f}")

**Training Model & Tuning**

In [ ]:
# PELATIHAN MODEL MACHINE LEARNING & HYPERPARAMETER TUNING
# 1. Training Regresi Logistik
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# 2. Training Random Forest dengan Hyperparameter Tuning (GridSearchCV)
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="recall",      # fokus meminimalkan False Negative (kasus ganas yang terlewat)
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
rf_best_model = rf_grid.best_estimator_

print("Proses Pelatihan Model Selesai!")
print(f"Hyperparameter Terbaik Random Forest: {rf_grid.best_params_}")
print(f"Recall CV terbaik                   : {rf_grid.best_score_:.4f}")

**Evaluasi Performa Model**

In [ ]:
# EVALUASI & PERBANDINGAN PERFORMA MODEL
# Prediksi Data Uji
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

y_pred_rf = rf_best_model.predict(X_test)
y_prob_rf = rf_best_model.predict_proba(X_test)[:, 1]

# Fungsi Evaluasi Metrik
def hitung_metrik(nama_model, y_true, y_pred, y_prob):
    return {
        "Model": nama_model,
        "Akurasi": accuracy_score(y_true, y_pred),
        "Presisi": precision_score(y_true, y_pred),
        "Recall (Sensitivitas)": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

tabel_hasil = pd.DataFrame([
    hitung_metrik("Regresi Logistik", y_test, y_pred_lr, y_prob_lr),
    hitung_metrik("Random Forest", y_test, y_pred_rf, y_prob_rf)
])

print("=== TABEL PERBANDINGAN METRIK EVALUASI ===")
display(tabel_hasil.round(4))

print("\n=== CLASSIFICATION REPORT: REGRESI LOGISTIK ===")
print(classification_report(y_test, y_pred_lr, target_names=["Jinak (B)", "Ganas (M)"], digits=3))
print("=== CLASSIFICATION REPORT: RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf, target_names=["Jinak (B)", "Ganas (M)"], digits=3))

**Visualisasi Evaluasi & Feature Importance**

In [ ]:
# VISUALISASI CONFUSION MATRIX, ROC CURVE, & FEATURE IMPORTANCE
label = ["Jinak (B)", "Ganas (M)"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt="d", cmap="Blues", ax=axes[0],
            cbar=False, xticklabels=label, yticklabels=label)
axes[0].set_title("Confusion Matrix - Regresi Logistik")
axes[0].set_xlabel("Prediksi")
axes[0].set_ylabel("Aktual")

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt="d", cmap="Greens", ax=axes[1],
            cbar=False, xticklabels=label, yticklabels=label)
axes[1].set_title("Confusion Matrix - Random Forest")
axes[1].set_xlabel("Prediksi")
axes[1].set_ylabel("Aktual")

plt.tight_layout()
plt.show()

# Plot Kurva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f"Regresi Logistik (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})", color="#2B6CB0")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})", color="#2F855A")
plt.plot([0, 1], [0, 1], "k--", label="Acak (Baseline)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Sensitivitas)")
plt.title("Kurva Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")
plt.show()

# Plot Feature Importance (Random Forest) - 10 fitur teratas
df_importance = (pd.DataFrame({"Fitur": X.columns, "Importance": rf_best_model.feature_importances_})
                 .sort_values("Importance", ascending=False))
top10 = df_importance.head(10).iloc[::-1]

plt.figure(figsize=(10, 5))
plt.barh(top10["Fitur"], top10["Importance"], color="#2F855A")
plt.title("10 Fitur Terpenting untuk Diagnosis (Feature Importance) - Random Forest")
plt.xlabel("Nilai Importance")
plt.tight_layout()
plt.show()

print("=== 10 FITUR TERPENTING ===")
display(df_importance.head(10).round(4).reset_index(drop=True))

## **Hasil dan Manfaat**
* **Hasil Kinerja Model** (data uji 114 pasien: 72 jinak, 42 ganas; `random_state=42`):

| Model | Akurasi | Presisi | Recall (Sensitivitas) | F1-Score | ROC-AUC |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Regresi Logistik | 96,49% | 97,50% | 92,86% | 95,12% | 0,996 |
| Random Forest | 97,37% | 100% | 92,86% | 96,30% | 0,995 |

  * **Random Forest** unggul tipis pada akurasi, presisi, dan F1-Score. Pada data uji, Random Forest **tidak menghasilkan *False Positive*** (tidak ada pasien jinak yang salah dinyatakan ganas), sedangkan Regresi Logistik menghasilkan 1 kasus.
  * **Recall kedua model sama (92,86%)**: masing-masing meloloskan **3 dari 42** kasus ganas sebagai jinak (*False Negative*). Untuk skrining medis, inilah titik lemah yang perlu diperhatikan, karena kasus ganas yang terlewat berdampak paling serius.
  * Kedua model sangat baik membedakan kelas secara umum (ROC-AUC ≈ 0,995–0,996).
  * Fitur dengan kontribusi terbesar menurut Random Forest adalah `concave points_worst`, `concave points_mean`, `area_worst`, `radius_worst`, dan `perimeter_mean`. Artinya, **ukuran dan kecembungan/ketidakteraturan kontur inti sel** paling menentukan pembeda sel ganas dan jinak.
  * Sebagai pembanding, penelitian aslinya melaporkan estimasi akurasi sekitar 97% (10-fold cross-validation) dan 100% pada 54 sampel pasien baru (Wolberg dkk., 1994). Hasil di atas berada pada rentang yang sejalan.
  * *Catatan:* angka dapat sedikit berbeda bila `random_state`, pembagian data, atau versi *library* berubah. Dengan hanya 114 data uji, selisih satu-dua pasien sudah mengubah persentase cukup besar.
* **Manfaat dalam Kehidupan Nyata (Dunia Medis):**
  1. **Pendukung Keputusan Klinis:** Memberi pembanding yang objektif beserta probabilitas keganasan bagi pemeriksa sebelum diagnosis ditetapkan.
  2. **Mengurangi Subjektivitas Pembacaan FNA:** Standar penilaian ciri inti sel menjadi lebih konsisten antar-pemeriksa.
  3. **Membantu Kasus Tidak Konklusif:** Berpotensi mengurangi rujukan biopsi bedah yang sebenarnya tidak perlu, sehingga menghemat biaya dan beban pasien.
  4. **Efisiensi Skrining:** Prediksi diperoleh dalam hitungan detik setelah fitur dari citra tersedia.
* **Batasan:** Model dilatih pada 569 sampel dari satu sumber data, sehingga perlu validasi eksternal pada data rumah sakit lain sebelum dipakai di praktik. Model ini alat bantu, **bukan pengganti** dokter/patolog.

## **Kesimpulan**
1. Algoritma **Random Forest** dan **Regresi Logistik** sama-sama efektif mengklasifikasikan benjolan payudara menjadi jinak atau ganas, dengan akurasi 96,5%–97,4% dan ROC-AUC di atas 0,99. Random Forest sedikit lebih unggul pada akurasi dan presisi.
2. Kedua model masih meloloskan 3 dari 42 kasus ganas (*recall* 92,86%). Untuk penerapan skrining, perbaikan yang layak dicoba antara lain menurunkan ambang probabilitas klasifikasi (*threshold tuning*), validasi silang yang lebih menyeluruh, dan pengujian pada data eksternal.
3. Fitur yang paling berpengaruh adalah ukuran inti sel (`radius`, `area`, `perimeter`) dan ketidakteraturan konturnya (`concave points`, `concavity`), sehingga hasil model sejalan dengan logika morfologi sel kanker.
4. Penerapan *Machine Learning* ini bermanfaat sebagai sistem pendukung keputusan klinis yang membuat diagnosis FNA lebih objektif, cepat, dan terukur.

## **Referensi**
1. **UCI Machine Learning Repository.** *Breast Cancer Wisconsin (Diagnostic)* [Dataset], didonasikan 1995. https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic
2. **Wolberg, W. H., Street, W. N., & Mangasarian, O. L. (1994).** *Machine learning techniques to diagnose breast cancer from image-processed nuclear features of fine needle aspirates.* Cancer Letters, 77(2), 163–171.
3. **Wolberg, W. H., Street, W. N., & Mangasarian, O. L. (1995).** *Image analysis and machine learning applied to breast cancer diagnosis and prognosis.* Analytical and Quantitative Cytology and Histology, 17(2), 77–87.
4. **Teague, M. W., dkk. (1997).** *Indeterminate fine-needle aspiration of the breast: Image analysis-assisted diagnosis.* Cancer, 81(2), 129–135.
5. **Breiman, L. (2001).** *Random Forests.* Machine Learning, 45(1), 5–32.
6. **Pedregosa, F., dkk. (2011).** *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning Research, 12, 2825–2830.